# 05 其他中间件（9 个官方中间件完整演示）

> 本文件逐一演示 langchain 1.2 中其余 **9 个官方中间件**，每个都包含：
> **功能介绍 / 参数列表 / 配置示例 / 使用效果 / 注意事项** 五部分，代码均可直接运行。
>
> 所有中间件均从 `langchain.agents.middleware` 导入（即官方文档 `middleware` 模块）。

---

## 0. 中间件总览表

| # | 中间件 | 类型 | 一句话功能 |
| --- | --- | --- | --- |
| 1 | `ModelCallLimitMiddleware` | 钩子型（before/after_model） | 统计并限制模型调用次数，超限后结束或抛错 |
| 2 | `ToolCallLimitMiddleware` | 钩子型（after_model） | 统计并限制工具调用次数，超限后可阻断/结束/抛错 |
| 3 | `ModelFallbackMiddleware` | 包装型（wrap_model_call） | 主模型失败时按序切换到备用模型 |
| 4 | `LLMToolSelectorMiddleware` | 包装型（wrap_model_call） | 用 LLM 挑选与问题相关的工具 |
| 5 | `ToolRetryMiddleware` | 包装型（wrap_tool_call） | 工具调用失败自动重试（指数退避） |
| 6 | `ModelRetryMiddleware` | 包装型（wrap_model_call） | 模型调用失败自动重试（指数退避） |
| 7 | `LLMToolEmulator` | 包装型（wrap_tool_call） | 用 LLM 仿真工具执行（测试用） |
| 8 | `ContextEditingMiddleware` | 包装型（wrap_model_call） | 超 token 阈值时自动清理旧工具结果 |
| 9 | `FilesystemFileSearchMiddleware` | 提供工具型（tools） | 为 Agent 注册 glob/grep 本地文件搜索工具 |

> **钩子型（hook）** 与 **包装型（wrap）** 的区别会在 06 文件中详细讲解。
> 这里只需记住：钩子型通过 `before_model / after_model` 在模型调用前后“插入节点”；
> 包装型通过 `wrap_model_call / wrap_tool_call` 把真实调用包起来。

---

In [1]:
# ============================================================
# 公共环境准备（本文件所有示例共用）
# ============================================================
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,          # 1 模型调用次数限制
    ToolCallLimitMiddleware,           # 2 工具调用次数限制
    ModelFallbackMiddleware,           # 3 模型回退
    LLMToolSelectorMiddleware,         # 4 工具选择
    ToolRetryMiddleware,               # 5 工具重试
    ModelRetryMiddleware,              # 6 模型重试
    LLMToolEmulator,                   # 7 工具仿真
    ContextEditingMiddleware,          # 8 上下文清理
    ClearToolUsesEdit,                 # 8 配套的清理策略配置
    FilesystemFileSearchMiddleware,    # 9 文件搜索
)
from langchain.agents.middleware.model_call_limit import ModelCallLimitExceededError
from langchain.agents.middleware.tool_call_limit import ToolCallLimitExceededError
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)

# 主模型（教程统一模型）
llm = init_chat_model(model="deepseek:deepseek-v4-flash",extra_body={"thinking":{"type":"disabled"}})
print("环境准备完成：", type(llm).__name__)

环境准备完成： ChatDeepSeek


In [2]:
# ============================================================
# 定义本文件各示例共用的演示工具
# ============================================================
@tool
def get_weather(city: str) -> str:
    """查询指定城市当前的天气情况。"""
    return f"{city}：晴，26℃，微风，空气质量良"


@tool
def get_time(city: str) -> str:
    """查询指定城市当前的本地时间。"""
    return f"{city} 当前时间：14:30"


@tool
def calculator(expression: str) -> str:
    """计算数学表达式并返回结果，例如 '1 + 2 * 3'。"""
    # 注意：eval 仅用于教学演示，生产环境请勿直接 eval 用户输入！
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:  # noqa: BLE001
        return f"计算失败：{e}"


@tool
def get_stock(code: str) -> str:
    """查询 A 股股票代码的最新价格。"""
    return f"{code} 最新价 12.34 元"


@tool
def search_news(keyword: str) -> str:
    """搜索与关键词相关的新闻标题列表。"""
    return f"【{keyword}】相关新闻：\n1. 今日发布新政策\n2. 行业迎来重大利好"

ALL_TOOLS = [get_weather, get_time, calculator, get_stock, search_news]
print("演示工具：", [t.name for t in ALL_TOOLS])

演示工具： ['get_weather', 'get_time', 'calculator', 'get_stock', 'search_news']


---

## 1. ModelCallLimitMiddleware —— 模型调用次数限制

### 功能介绍
统计 Agent 执行过程中**模型被调用的次数**，当达到设定上限时，可以选择：
- `"end"`：立即跳到 Agent 结束节点，并注入一条说明超限的 AI 消息（Agent 正常结束）；
- `"error"`：抛出 `ModelCallLimitExceededError` 异常，终止执行。

计数分为两个维度：
- **thread（会话级）**：跨多次 `invoke` 持久累加（需要 checkpointer 配合，否则每次运行独立）；
- **run（单次运行级）**：仅统计单次 `invoke` 内的调用次数。

### 参数列表
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `thread_limit` | `int \| None` | `None` | 单会话（thread）最多模型调用次数，`None` 不限制 |
| `run_limit` | `int \| None` | `None` | 单次运行（run）最多模型调用次数，`None` 不限制 |
| `exit_behavior` | `"end" \| "error"` | `"end"` | 超限后的行为 |

### 注意事项
- ⚠️ `thread_limit` 与 `run_limit` **至少要设置一个**，否则构造时报 `ValueError`。
- ⚠️ 计数发生在 `after_model`，检查发生在下一次 `before_model`（即“第 N 次调用后，第 N+1 次调用前”拦截）。
- 💡 判断条件：当 `count >= limit` 时视为超限（即第 `limit` 次调用完成后即触发）。

### 示例1：超限后正常结束（end）

In [4]:
# 限制：单次运行最多 3 次模型调用，超限后 Agent 直接结束
limiter = ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")

agent = create_agent(
    model=llm,
    tools=[get_weather],          # 多轮工具调用会消耗多轮模型调用
    middleware=[limiter],
)

resp = await agent.ainvoke({
    "messages": [
        HumanMessage(content="请依次查询北京、上海、广州三个城市的天气，然后汇总")
    ]
})

# 观察：最后一次 AI 消息是“Model call limits exceeded: run limit (3/3)”之类的说明
for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

请依次查询北京、上海、广州三个城市的天气，然后汇总
================================== Ai Message ==================================

I'll query the weather for all three cities in parallel.
Tool Calls:
  get_weather (call_00_YnwvnN93ldVsl12fOB4e1770)
 Call ID: call_00_YnwvnN93ldVsl12fOB4e1770
  Args:
    city: 北京
  get_weather (call_01_h1IW7A7HlDpEvJoomDCx0632)
 Call ID: call_01_h1IW7A7HlDpEvJoomDCx0632
  Args:
    city: 上海
  get_weather (call_02_WUKoouia90LQrfgZqXzk5384)
 Call ID: call_02_WUKoouia90LQrfgZqXzk5384
  Args:
    city: 广州
================================= Tool Message =================================
Name: get_weather

北京：晴，26℃，微风，空气质量良
================================= Tool Message =================================
Name: get_weather

上海：晴，26℃，微风，空气质量良
================================= Tool Message =================================
Name: get_weather

广州：晴，26℃，微风，空气质量良
================================== Ai Message ==

### 示例2：超限后抛出异常（error）

In [ ]:
# 限制：单次运行最多 2 次模型调用，超限后抛异常
limiter_err = ModelCallLimitMiddleware(run_limit=2, exit_behavior="error")

agent_err = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[limiter_err],
)

try:
    await agent_err.ainvoke({
        "messages": [
            HumanMessage(content="请查询北京的天气")
        ]
    })
except ModelCallLimitExceededError as e:
    # 捕获并打印异常，异常信息中会包含具体的超限明细
    rprint(f"[bold red]捕获到 ModelCallLimitExceededError：[/]{e}")

---

## 2. ToolCallLimitMiddleware —— 工具调用次数限制

### 功能介绍
统计 Agent 执行过程中**工具被调用的次数**，达到上限后有三种行为：
- `"continue"`（默认）：**阻断超限的工具调用**，注入一条 `status="error"` 的 `ToolMessage`
  （如 “Tool call limit exceeded. Do not make additional tool calls.”），让模型看到错误后继续工作；
- `"error"`：抛出 `ToolCallLimitExceededError`；
- `"end"`：立即结束（仅适用于单工具调用超限，存在并行工具调用时抛 `NotImplementedError`）。

支持两种统计维度（thread / run），并支持**按工具名分别统计**：
计数是一个 `dict[str, int]`，特殊键 `"__all__"` 表示对所有工具的总计数。

### 参数列表
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `tool_name` | `str \| None` | `None` | 只统计指定名称的工具；`None` 表示统计所有工具（键为 `__all__`） |
| `thread_limit` | `int \| None` | `None` | 单会话最多工具调用次数 |
| `run_limit` | `int \| None` | `None` | 单次运行最多工具调用次数 |
| `exit_behavior` | `"continue" \| "error" \| "end"` | `"continue"` | 超限后的行为 |

### 注意事项
- ⚠️ 两个 limit **至少要设置一个**。
- ⚠️ `run_limit` **不能大于** `thread_limit`（构造时校验，避免逻辑矛盾）。
- ⚠️ `exit_behavior="end"` 时，若本轮还有其他工具的并行调用，会抛 `NotImplementedError`，
  请改用 `continue` 或 `error`。
- 💡 被阻断的工具调用**不会计入 thread 计数**，但会计入 run 计数（表示“本次运行尝试过”）。

### 示例1：超限后阻断该工具、模型继续（continue，默认）

In [ ]:
# 限制：单次运行最多 2 次工具调用，超限后阻断继续执行
limiter_tool = ToolCallLimitMiddleware(run_limit=2, exit_behavior="continue")

agent_tool = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[limiter_tool],
)

resp_tool = await agent_tool.ainvoke({
    "messages": [
        HumanMessage(content="请依次查询北京、上海、广州、深圳四个城市的天气")
    ]
})

# 观察：第 3、4 个城市的工具调用会被替换为 error ToolMessage，
# 模型读到错误后通常直接给出“无法继续查询”之类的收尾回答
for msg in resp_tool["messages"]:
    msg.pretty_print()

### 示例2：超限后抛出异常（error）

In [ ]:
# 严格限制：单次运行最多 1 次工具调用，超限即抛错
limiter_tool_err = ToolCallLimitMiddleware(run_limit=1, exit_behavior="error")

agent_tool_err = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[limiter_tool_err],
)

try:
    await agent_tool_err.ainvoke({
        "messages": [
            HumanMessage(content="请分别查询北京和上海的天气")
        ]
    })
except ToolCallLimitExceededError as e:
    rprint(f"[bold red]捕获到 ToolCallLimitExceededError：[/]{e}")

---

## 3. ModelFallbackMiddleware —— 模型故障自动回退

### 功能介绍
当**主模型**（`create_agent(model=...)` 中指定的模型）调用**抛异常**时，自动按顺序尝试**备用模型**，
直到某个模型成功，或所有模型都失败（此时重新抛出最后一次异常）。

典型场景：主模型服务不可用 / 限流 / 超时，希望自动切换到一个兜底模型继续服务。

### 参数列表
```python
ModelFallbackMiddleware(
    first_model: str | BaseChatModel,        # 第一个备用模型（必填）
    *additional_models: str | BaseChatModel, # 其余备用模型，按顺序排列（可选）
)
```
> 注意：这里是**位置参数**（不是关键字参数），多个备用模型按传入顺序依次尝试。

### 注意事项
- ⚠️ 只对**模型调用抛出的异常**生效；工具执行失败不归它管（用 `ToolRetryMiddleware`）。
- ⚠️ 主模型由 `create_agent(model=...)` 指定；中间件里的参数只是“备用梯队”。
- 💡 与 `ModelRetryMiddleware` 组合时要注意 `on_failure` 行为（详见 06 文件）。

### 示例：主模型必然失败 → 自动回退到备用模型
下面自定义一个“永远失败”的测试模型充当主模型，用来真实演示回退生效。

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage
from langchain_core.outputs import ChatResult


class AlwaysFailChatModel(BaseChatModel):
    """一个任何调用都会抛异常的主模型，用于演示回退机制。"""

    @property
    def _llm_type(self) -> str:
        return "always-fail"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        # 真实实现中，这里就是模型调用；这里直接模拟“服务不可用”
        raise RuntimeError("主模型服务不可用（模拟故障）")


# 备用模型梯队：主模型失败后 → 尝试 deepseek
fallback = ModelFallbackMiddleware(
    "deepseek:deepseek-v4-flash",   # 第一个备用模型
    # 还可以继续追加： "deepseek:deepseek-chat", ...
)

agent_fb = create_agent(
    model=AlwaysFailChatModel(),   # 主模型：必然失败
    middleware=[fallback],         # 挂上回退中间件
)

resp_fb = await agent_fb.ainvoke({
    "messages": [
        HumanMessage(content="你好，请简单介绍一下你自己")
    ]
})

# 主模型失败了，但 Agent 仍成功返回 —— 说明回退已生效
resp_fb["messages"][-1].pretty_print()

---

## 4. LLMToolSelectorMiddleware —— LLM 工具选择

### 功能介绍
用 LLM 从大量工具中挑选与当前问题最相关的工具，只把选中工具交给主模型。
这里给出一个可直接运行的示例，更多细节可参考官方文档。

### 参数速览
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `model` | `str \| BaseChatModel \| None` | `None` | 做选择的模型；`None` 复用主模型 |
| `system_prompt` | `str` | 英文默认提示词 | 选择模型的指令 |
| `max_tools` | `int \| None` | `None` | 最多选几个工具 |
| `always_include` | `list[str] \| None` | `None` | 无条件保留的工具（不计入 max_tools） |

### 示例

In [ ]:
selector = LLMToolSelectorMiddleware(
    max_tools=2,                          # 最多选 2 个
    always_include=["get_time"],          # 时间工具永远保留
)

agent_sel = create_agent(
    model=llm,
    tools=ALL_TOOLS,   # 5 个工具，但主模型最多只看到 3 个（2 选中 + 1 保留）
    middleware=[selector],
)

resp_sel = await agent_sel.ainvoke({
    "messages": [
        HumanMessage(content="现在北京时间几点？今天天气适合跑步吗？")
    ]
})

for msg in resp_sel["messages"]:
    msg.pretty_print()

---

## 5. ToolRetryMiddleware —— 工具调用失败自动重试

### 功能介绍
当**工具执行抛出异常**时，自动按指数退避策略重试，直到成功或达到最大次数。
重试耗尽后的处理方式（`on_failure`）有三种：
- `"continue"`（默认）：返回一条 `status="error"` 的 `ToolMessage`，把错误交给模型消化；
- `"error"`：直接抛出原始异常，终止 Agent；
- **自定义 callable**：`fn(exc) -> str`，把异常转成自定义错误文案放进 `ToolMessage`。

### 参数列表
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `max_retries` | `int` | `2` | 初始调用之外的最大重试次数（须 `>=0`） |
| `tools` | `list[BaseTool \| str] \| None` | `None` | 只对指定工具生效；`None` 对所有工具生效 |
| `retry_on` | `tuple[Exception] \| Callable` | `(Exception,)` | 哪些异常需要重试；可传自定义判断函数 |
| `on_failure` | `"continue" \| "error" \| Callable` | `"continue"` | 重试耗尽后的行为 |
| `backoff_factor` | `float` | `2.0` | 退避倍数：每次等待 `initial_delay * backoff_factor ** n` 秒；设 `0.0` 为固定延迟 |
| `initial_delay` | `float` | `1.0` | 首次重试前的等待秒数 |
| `max_delay` | `float` | `60.0` | 单次最大等待秒数（封顶退避增长） |
| `jitter` | `bool` | `True` | 是否加 ±25% 随机抖动，避免惊群 |

### 注意事项
- ⚠️ `on_failure` 的旧值 `"raise"` / `"return_message"` 已弃用，请用 `"error"` / `"continue"`。
- 💡 `retry_on` 传 tuple 时只重试列出的异常类型；传 callable 时返回 `True` 才重试。

### 示例1：第一次调用失败，重试后成功

In [ ]:
# 用一个“第一次必失败”的工具模拟网络抖动
_state = {"calls": 0}


@tool
def flaky_weather(city: str) -> str:
    """查询天气，第一次调用会抛异常（模拟网络抖动）。"""
    _state["calls"] += 1
    if _state["calls"] == 1:
        raise ConnectionError("模拟第一次网络连接异常")
    return f"{city} 天气：晴，26℃"


retry_tool = ToolRetryMiddleware(
    max_retries=2,
    initial_delay=0.1,   # 教学中把等待调小，避免长时间卡住
    backoff_factor=1.0,  # 固定延迟
)

agent_rt = create_agent(
    model=llm,
    tools=[flaky_weather],
    middleware=[retry_tool],
)

resp_rt = await agent_rt.ainvoke({
    "messages": [
        HumanMessage(content="查询北京天气")
    ]
})

rprint(f"[bold green]flaky_weather 实际被调用次数：[/]{_state['calls']}（应为 2）")
resp_rt["messages"][-1].pretty_print()

### 示例2：只重试特定异常 + 自定义失败文案（配置演示）

In [ ]:
from requests.exceptions import RequestException, Timeout


def format_db_error(exc: Exception) -> str:
    """把数据库异常转成用户友好的错误提示。"""
    return "数据库暂时不可用，请稍后重试。"


# 仅对 RequestException / Timeout 重试；重试耗尽后返回自定义错误消息
retry_tool2 = ToolRetryMiddleware(
    max_retries=4,
    tools=["get_weather"],              # 只作用于名为 get_weather 的工具
    retry_on=(RequestException, Timeout),  # 只重试网络类异常
    on_failure=format_db_error,         # 自定义失败文案
    backoff_factor=1.5,
)

rprint("ToolRetryMiddleware 高级配置示例（仅展示配置，不运行）：")
rprint(retry_tool2)

---

## 6. ModelRetryMiddleware —— 模型调用失败自动重试

### 功能介绍
与 `ToolRetryMiddleware` 对称，作用于**模型调用**：模型调用抛异常时自动重试，
支持指数退避、异常过滤与失败策略。适用于：模型服务限流、超时、5xx 错误等。

### 参数列表
与 `ToolRetryMiddleware` 一致，但**没有 `tools` 参数**（它作用的是模型而非工具）：

| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `max_retries` | `int` | `2` | 最大重试次数 |
| `retry_on` | `tuple[Exception] \| Callable` | `(Exception,)` | 重试条件 |
| `on_failure` | `"continue" \| "error" \| Callable` | `"continue"` | 重试耗尽后的行为 |
| `backoff_factor` / `initial_delay` / `max_delay` / `jitter` | 同工具版 | — | 退避参数 |

> `on_failure="continue"` 时，耗尽后返回一条“Model call failed after N attempts...”的 `AIMessage`，
> Agent 会带着这条错误消息继续；若想让异常继续向上抛（例如配合 `ModelFallbackMiddleware`），请设 `"error"`。

### 示例1：模型第一次调用失败，重试后成功

In [ ]:
class FlakyChatModel(BaseChatModel):
    """第一次调用抛异常，之后委托给真实模型执行。"""

    underlying: BaseChatModel  # 真正干活的模型
    calls: int = 0             # 统计调用次数

    @property
    def _llm_type(self) -> str:
        return "flaky"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        self.calls += 1
        if self.calls == 1:
            raise ConnectionError("模拟第一次模型调用失败")
        # 委托给底层真实模型（deepseek）生成结果
        return self.underlying._generate(
            messages, stop=stop, run_manager=run_manager, **kwargs
        )


# 用真实 deepseek 模型作为底层，外面包一层“第一次必失败”
flaky_model = FlakyChatModel(underlying=llm)

retry_model = ModelRetryMiddleware(
    max_retries=2,
    initial_delay=0.1,
    backoff_factor=1.0,
)

agent_rm = create_agent(
    model=flaky_model,
    middleware=[retry_model],
)

resp_rm = await agent_rm.ainvoke({
    "messages": [
        HumanMessage(content="你好")
    ]
})

rprint(f"[bold green]模型实际被调用次数：[/]{flaky_model.calls}（应为 2）")
resp_rm["messages"][-1].pretty_print()

### 示例2：只重试限流类异常（配置演示）

In [ ]:
from anthropic import RateLimitError
from openai import APITimeoutError

# 生产中常见用法：只对限流/超时做重试，其他错误直接失败
retry_model2 = ModelRetryMiddleware(
    max_retries=4,
    retry_on=(APITimeoutError, RateLimitError),
    backoff_factor=1.5,
    initial_delay=1.0,
    max_delay=30.0,
)

rprint("ModelRetryMiddleware 生产配置示例（仅展示配置，不运行）：")
rprint(retry_model2)

---

## 7. LLMToolEmulator —— 用 LLM 仿真工具执行（测试利器）

### 功能介绍
**不真正执行工具**，而是让一个 LLM 根据“工具名 + 工具描述 + 调用参数”**伪造一个逼真的工具返回结果**。
主要用于：
- **测试 / 演示**：没有真实后端（如真实天气 API）时，先跑通整个 Agent 流程；
- **故障演练**：让工具永远返回可控的假数据，方便调试；
- **成本控制**：避免频繁调用昂贵的真实工具。

### 参数列表
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `tools` | `list[str \| BaseTool] \| None` | `None` | 要仿真的工具（名字或实例）。`None` = 全部仿真；`[]` = 不仿真 |
| `model` | `str \| BaseChatModel \| None` | `"anthropic:claude-sonnet-4-5-20250929"` | 做仿真的模型 |

> ⚠️ `model` 默认值是 Anthropic 的模型（需要 `ANTHROPIC_API_KEY`）。本项目使用 DeepSeek，
> 因此**必须显式传 `model="deepseek:deepseek-v4-flash"`**。

### 示例：仿真一个“真实实现故意缺失”的计算器工具
下面工具的真实实现会抛 `NotImplementedError` —— 如果 Agent 仍能成功回答，
说明工具被 LLM 仿真了，而非真实执行。

In [ ]:
@tool
def calculator_sim(expression: str) -> str:
    """计算数学表达式。注意：本工具的真实实现故意缺失，应由 LLMToolEmulator 仿真。"""
    raise NotImplementedError("真实工具不应被执行——此工具应由 LLMToolEmulator 仿真")


emulator = LLMToolEmulator(
    tools=[calculator_sim],              # 只仿真这个工具
    model="deepseek:deepseek-v4-flash",  # 必须覆盖默认的 Anthropic 模型
)

agent_em = create_agent(
    model=llm,
    tools=[calculator_sim],
    middleware=[emulator],
)

resp_em = await agent_em.ainvoke({
    "messages": [
        HumanMessage(content="帮我算一下 12345 乘以 6789 等于多少？")
    ]
})

# 若能正常返回，说明工具调用被 LLM 仿真（结果由模型推算/估算给出）
resp_em["messages"][-1].pretty_print()

---

## 8. ContextEditingMiddleware —— 上下文超限自动清理

### 功能介绍
当**输入消息总 token 数超过阈值**时，自动把**较早的工具执行结果**替换为占位符（默认 `[cleared]`），
从而给后续对话腾出上下文空间。对应 Anthropic 的 `clear_tool_uses` 能力。

### 参数列表

**`ContextEditingMiddleware`**：
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `edits` | `Iterable[ContextEdit]` | `(ClearToolUsesEdit(),)` | 一组清理策略 |
| `token_count_method` | `"approximate" \| "model"` | `"approximate"` | 计数方式：近似（快）或模型精确计数（慢但准） |

**`ClearToolUsesEdit`**（默认策略）：
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `trigger` | `int` | `100_000` | 触发清理的 token 阈值 |
| `clear_at_least` | `int` | `0` | 至少要回收多少 token；`0` = 清一条就算完成 |
| `keep` | `int` | `3` | 保留最近的 N 条工具结果（其余清掉） |
| `clear_tool_inputs` | `bool` | `False` | 是否同时清空被清理工具对应的 AI 调用参数 |
| `exclude_tools` | `Sequence[str]` | `()` | 豁免清理的工具名列表 |
| `placeholder` | `str` | `"[cleared]"` | 占位符文本 |

### 注意事项
- ⚠️ 只有**比 `keep` 更早**的 `ToolMessage` 才会被清理；`exclude_tools` 中的工具不清理。
- 💡 清理是**深拷贝**后再替换，不会污染原始消息。
- 💡 本项目为了演示效果，把 `trigger` 调得很小（几百 token），真实场景建议按上下文窗口设置。

### 示例：多次工具调用后，旧结果被替换为占位符

In [ ]:
context_mw = ContextEditingMiddleware(
    edits=[
        ClearToolUsesEdit(
            trigger=250,        # 消息累计超过约 250 token 就触发清理（演示用，调小）
            keep=1,             # 只保留最近 1 条工具结果
            placeholder="[已清理]",  # 被清理内容的占位符
        )
    ],
)

agent_ctx = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[context_mw],
)

resp_ctx = await agent_ctx.ainvoke({
    "messages": [
        HumanMessage(content="请依次查询北京、上海、广州、深圳、杭州五个城市的天气，并逐条汇报结果")
    ]
})

# 打印所有 ToolMessage，观察较老的几条内容是否已被替换为 [已清理]
for msg in resp_ctx["messages"]:
    if isinstance(msg, ToolMessage):
        rprint(f"[ToolMessage] {msg.name}: {msg.content}")

# 若没有出现 [已清理]，说明对话累计 token 未超过 250，
# 可把 trigger 调小（例如 150）再试一次。

---

## 9. FilesystemFileSearchMiddleware —— 本地文件搜索工具

### 功能介绍
为 Agent **自动注册两个文件搜索工具**，让 Agent 具备检索本地文件系统的能力：
- **`glob_search`**：按路径/通配模式快速查找文件（如 `**/*.py`）；
- **`grep_search`**：按正则表达式搜索文件内容，支持 `files_with_matches` / `content` / `count` 三种输出模式。

内部默认优先使用 `ripgrep`（更快），不可用时自动回退到纯 Python 实现。

### 参数列表
| 参数 | 类型 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `root_path` | `str` | **必填** | 允许搜索的根目录（会做路径穿越防护，越界报错） |
| `use_ripgrep` | `bool` | `True` | 是否优先用 ripgrep |
| `max_file_size_mb` | `int` | `10` | 参与内容搜索的最大文件大小（MB） |

### 注意事项
- ⚠️ 工具会做**路径穿越防护**：`..` / `~` 等越界路径会被拒绝，且结果路径都会被转成相对 root 的虚拟路径。
- ⚠️ 该中间件属于“提供工具型”：它把 `glob_search` / `grep_search` 挂在 `middleware.tools` 上，
  因此 `create_agent(tools=[...])` 可以不传工具。

### 示例：让 Agent 搜索本教程项目中的文件与内容

In [ ]:
# 让 Agent 可以搜索整个教程项目目录
fs_mw = FilesystemFileSearchMiddleware(
    root_path=r"d:\code\python\langchain1.2_tutorial",  # 搜索根目录
    use_ripgrep=True,
)

agent_fs = create_agent(
    model=llm,
    tools=[],        # 工具由中间件自动注册
    middleware=[fs_mw],
)

resp_fs = await agent_fs.ainvoke({
    "messages": [
        HumanMessage(
            content="请帮我在 chapter08-Middleware中间件 目录下查找所有以 0 开头的 .ipynb 文件；"
                    "再在 04-TodoListMiddleware中间件.ipynb 文件里搜索'TodoListMiddleware'，"
                    "告诉我一共出现几次。"
        )
    ]
})

resp_fs["messages"][-1].pretty_print()

---

## 总结

| 中间件 | 一句话记忆点 |
| --- | --- |
| `ModelCallLimitMiddleware` | 模型调用次数上限，`end` 温和 / `error` 严格 |
| `ToolCallLimitMiddleware` | 工具调用次数上限，可按工具名单独限流 |
| `ModelFallbackMiddleware` | 主模型挂了自动切备用梯队 |
| `LLMToolSelectorMiddleware` | 用 LLM 动态裁剪工具列表（见 04） |
| `ToolRetryMiddleware` | 工具失败自动重试（指数退避） |
| `ModelRetryMiddleware` | 模型失败自动重试（指数退避） |
| `LLMToolEmulator` | 用 LLM 伪造工具结果，测试利器 |
| `ContextEditingMiddleware` | 上下文太长时清掉旧工具结果 |
| `FilesystemFileSearchMiddleware` | 给 Agent 装 glob/grep 本地搜索能力 |

> 本文件完。接下来请学习 **06 文件**：多个中间件如何组合、以及执行顺序规则。